# Attention Profiling for Flux 2 Klein-4B

This notebook implements **Phase 1** of the attention profiling and masking plan.

**Goals:**
1. Extract attention weight metrics from every attention layer at selected denoising timesteps
2. Visualize attention heatmaps and quadrant patterns (T→T, T→I, I→T, I→I)
3. Analyze attention entropy, sparsity, and top-k concentration across blocks and timesteps

**Architecture:**
- 5 double-stream blocks (`Flux2TransformerBlock`) with joint attention
- 20 single-stream blocks (`Flux2SingleTransformerBlock`) with parallel self-attention
- 24 attention heads, 128 dim per head


## 1. Setup & Model Loading

In [ ]:
# Uncomment and adjust these lines for Google Colab / Drive setup
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/diffusers_csc2210
# !pip install -e .


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import defaultdict
from IPython.display import display, HTML

from diffusers import Flux2KleinPipeline, PipelineQuantizationConfig
from diffusers.models.transformers.transformer_flux2 import (
    AttentionProfileEntry,
    AttentionProfilingStore,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device}, dtype: {dtype}")


In [ ]:
# Load the model — adjust cache_dir to your local path or use HuggingFace Hub
# Option A: From local cache
# cache_dir = "/path/to/your/flux2-klein-4B/snapshot"
# pipe = Flux2KleinPipeline.from_pretrained(cache_dir, torch_dtype=dtype)

# Option B: From HuggingFace Hub with quantization
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype
)
quantization_config = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs=bnb_config.to_dict()
)

pipe = Flux2KleinPipeline.from_pretrained(
    "black-forest-labs/FLUX.2-klein-4B",
    quantization_config=quantization_config,
    torch_dtype=dtype,
)
pipe.enable_model_cpu_offload()
print(f"Model loaded. Double-stream blocks: {len(pipe.transformer.transformer_blocks)}, "
      f"Single-stream blocks: {len(pipe.transformer.single_transformer_blocks)}")


## 2. Enable Attention Profiling

This swaps all attention processors to profiling variants that capture metrics
at each block and timestep, while preserving the original model output.

In [ ]:
# Enable profiling mode on the transformer
# store_full_weights=False to save memory (only summary statistics are stored)
pipe.transformer.set_profiling_mode(
    enabled=True,
    store_full_weights=False,  # Set True to capture full [B,H,S,S] attention matrices
    sparsity_threshold=0.01,
    topk=32,
)
print("Attention profiling enabled.")


## 3. Run Inference with Profiling

We run inference with diverse prompts to capture attention patterns across different inputs.

In [ ]:
# Define diverse prompts for profiling
prompts = [
    "a photorealistic cat sitting on a wooden table",
    "a futuristic cityscape at sunset with flying cars",
    "abstract swirling colors in a cosmic nebula",
]

# Run inference for each prompt
all_entries = []
for i, prompt in enumerate(prompts):
    pipe.transformer.clear_attention_profile_data()
    print(f"\nPrompt {i+1}/{len(prompts)}: '{prompt}'")
    
    image = pipe(
        prompt=prompt,
        height=512,
        width=512,
        guidance_scale=1.0,
        num_inference_steps=4,
        generator=torch.Generator(device=device).manual_seed(42),
    ).images[0]
    
    entries = pipe.transformer.get_attention_profile_data()
    print(f"  Captured {len(entries)} profiling entries")
    
    # Tag entries with prompt index
    for e in entries:
        e.prompt_index = i  # Monkey-patch for tracking
    all_entries.extend(entries)
    
    # Display the generated image
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title(f"Prompt {i+1}")
    plt.axis("off")
    plt.show()

print(f"\nTotal profiling entries: {len(all_entries)}")


## 4. Organize Profiling Data

Group entries by block type, block index, and timestep for analysis.

In [ ]:
# Organize entries into a structured dictionary
# Key: (block_type, block_index) -> list of entries across timesteps and prompts
entries_by_block = defaultdict(list)
entries_by_timestep = defaultdict(list)
unique_timesteps = sorted(set(e.timestep for e in all_entries))

for e in all_entries:
    entries_by_block[(e.block_type, e.block_index)].append(e)
    entries_by_timestep[round(e.timestep, 4)].append(e)

# Print summary
print(f"Unique timesteps: {len(unique_timesteps)}")
print(f"Unique blocks: {len(entries_by_block)}")
for key in sorted(entries_by_block.keys()):
    print(f"  {key[0]} block {key[1]}: {len(entries_by_block[key])} entries")


## 5. Quadrant Analysis

Analyze the four attention quadrants (T→T, T→I, I→T, I→I) across block depth.

The attention matrix for a sequence of [text_tokens; image_tokens] has four regions:
```
         text_K    img_K
text_Q [ T→T      T→I   ]
img_Q  [ I→T      I→I   ]
```

In [ ]:
def plot_quadrant_analysis(entries, title_suffix=""):
    """Plot mean attention weight in each quadrant across block depth."""
    # Separate double and single stream
    double_entries = [e for e in entries if e.block_type == "double"]
    single_entries = [e for e in entries if e.block_type == "single"]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    for ax, block_entries, block_label in [
        (axes[0], double_entries, "Double-Stream (Joint Attn)"),
        (axes[1], single_entries, "Single-Stream (Self-Attn)"),
    ]:
        if not block_entries:
            ax.set_title(f"{block_label} — No data")
            continue
        
        # Group by block index, average across timesteps and prompts
        block_indices = sorted(set(e.block_index for e in block_entries))
        quadrants = {"T→T": [], "T→I": [], "I→T": [], "I→I": []}
        
        for idx in block_indices:
            block_e = [e for e in block_entries if e.block_index == idx]
            quadrants["T→T"].append(np.mean([e.mean_weight_tt for e in block_e]))
            quadrants["T→I"].append(np.mean([e.mean_weight_ti for e in block_e]))
            quadrants["I→T"].append(np.mean([e.mean_weight_it for e in block_e]))
            quadrants["I→I"].append(np.mean([e.mean_weight_ii for e in block_e]))
        
        x = np.arange(len(block_indices))
        width = 0.2
        colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]
        
        for i, (label, values) in enumerate(quadrants.items()):
            ax.bar(x + i * width, values, width, label=label, color=colors[i])
        
        ax.set_xlabel("Block Index")
        ax.set_ylabel("Mean Attention Weight")
        ax.set_title(f"{block_label}{title_suffix}")
        ax.set_xticks(x + 1.5 * width)
        ax.set_xticklabels(block_indices)
        ax.legend()
    
    plt.tight_layout()
    plt.show()

plot_quadrant_analysis(all_entries, " — Averaged Across All Prompts & Timesteps")


In [ ]:
# Quadrant analysis per timestep
for t in unique_timesteps[::max(1, len(unique_timesteps) // 4)]:
    t_round = round(t, 4)
    t_entries = entries_by_timestep.get(t_round, [])
    if t_entries:
        plot_quadrant_analysis(t_entries, f" — Timestep t={t_round:.4f}")


## 6. Entropy & Sparsity Profiles

Plot per-block attention entropy and sparsity to identify which blocks have 
exploitable sparse attention patterns.

In [ ]:
def plot_entropy_sparsity(entries, title_suffix=""):
    """Plot per-block entropy and sparsity."""
    double_entries = [e for e in entries if e.block_type == "double"]
    single_entries = [e for e in entries if e.block_type == "single"]
    
    # Combine into a single block axis: double blocks first, then single blocks
    all_blocks = []
    all_labels = []
    
    for e_list, prefix in [(double_entries, "D"), (single_entries, "S")]:
        block_indices = sorted(set(e.block_index for e in e_list))
        for idx in block_indices:
            block_e = [e for e in e_list if e.block_index == idx]
            mean_entropy = torch.stack([e.per_head_entropy for e in block_e]).mean().item()
            mean_sparsity = torch.stack([e.per_head_sparsity for e in block_e]).mean().item()
            mean_topk = torch.stack([e.per_head_topk_concentration for e in block_e]).mean().item()
            all_blocks.append((mean_entropy, mean_sparsity, mean_topk))
            all_labels.append(f"{prefix}{idx}")
    
    if not all_blocks:
        print("No data to plot.")
        return
    
    entropies, sparsities, topks = zip(*all_blocks)
    x = np.arange(len(all_labels))
    
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # Entropy plot
    ax1.bar(x, entropies, color="#2196F3", alpha=0.8)
    ax1.set_ylabel("Mean Entropy")
    ax1.set_title(f"Per-Block Attention Entropy{title_suffix}")
    ax1.axvline(x=len([e for e in double_entries if True]) // max(1, len(set(e.block_index for e in double_entries))) - 0.5 if double_entries else -1,
               color="gray", linestyle="--", alpha=0.5, label="Double→Single boundary")
    
    # Sparsity plot
    ax2.bar(x, sparsities, color="#F44336", alpha=0.8)
    ax2.set_ylabel(f"Sparsity (frac < {entries[0].num_text_tokens if entries else 'N/A'})")
    ax2.set_title(f"Per-Block Attention Sparsity{title_suffix}")
    
    # Top-k concentration plot
    ax3.bar(x, topks, color="#4CAF50", alpha=0.8)
    ax3.set_ylabel("Top-k Concentration")
    ax3.set_title(f"Per-Block Top-k Attention Concentration{title_suffix}")
    ax3.set_xticks(x)
    ax3.set_xticklabels(all_labels, rotation=45, ha="right")
    ax3.set_xlabel("Block (D=Double, S=Single)")
    
    plt.tight_layout()
    plt.show()

plot_entropy_sparsity(all_entries, " — Averaged Across All Prompts & Timesteps")


## 7. Per-Head Analysis

Examine whether individual attention heads specialize in different patterns.

In [ ]:
def plot_per_head_analysis(entries, block_type="double", block_index=0):
    """Plot per-head entropy and sparsity for a specific block."""
    block_entries = [e for e in entries if e.block_type == block_type and e.block_index == block_index]
    if not block_entries:
        print(f"No data for {block_type} block {block_index}")
        return
    
    # Average across timesteps and prompts
    entropies = torch.stack([e.per_head_entropy for e in block_entries]).mean(dim=0)  # [H]
    sparsities = torch.stack([e.per_head_sparsity for e in block_entries]).mean(dim=0)  # [H]
    topks = torch.stack([e.per_head_topk_concentration for e in block_entries]).mean(dim=0)  # [H]
    
    num_heads = entropies.shape[0]
    x = np.arange(num_heads)
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))
    
    ax1.bar(x, entropies.numpy(), color="#2196F3")
    ax1.set_xlabel("Head Index")
    ax1.set_ylabel("Entropy")
    ax1.set_title(f"Per-Head Entropy — {block_type} block {block_index}")
    
    ax2.bar(x, sparsities.numpy(), color="#F44336")
    ax2.set_xlabel("Head Index")
    ax2.set_ylabel("Sparsity")
    ax2.set_title(f"Per-Head Sparsity — {block_type} block {block_index}")
    
    ax3.bar(x, topks.numpy(), color="#4CAF50")
    ax3.set_xlabel("Head Index")
    ax3.set_ylabel("Top-k Concentration")
    ax3.set_title(f"Per-Head Top-k — {block_type} block {block_index}")
    
    plt.tight_layout()
    plt.show()

# Plot per-head analysis for selected blocks
# Double-stream: first and last
num_double = len(pipe.transformer.transformer_blocks)
num_single = len(pipe.transformer.single_transformer_blocks)

print("=== Double-Stream Blocks ===")
for idx in [0, num_double - 1]:
    plot_per_head_analysis(all_entries, "double", idx)

print("\n=== Single-Stream Blocks ===")
for idx in [0, num_single // 2, num_single - 1]:
    plot_per_head_analysis(all_entries, "single", idx)


## 8. Temporal Evolution

How do attention patterns change across denoising timesteps?

In [ ]:
def plot_temporal_evolution(entries):
    """Plot how entropy and quadrant ratios change across timesteps."""
    timesteps = sorted(set(e.timestep for e in entries))
    if len(timesteps) < 2:
        print("Need at least 2 timesteps for temporal evolution plot.")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    for ax_row, block_type, label in [
        (axes[0], "double", "Double-Stream"),
        (axes[1], "single", "Single-Stream"),
    ]:
        block_entries = [e for e in entries if e.block_type == block_type]
        if not block_entries:
            continue
        
        block_indices = sorted(set(e.block_index for e in block_entries))
        
        # Entropy evolution
        ax = ax_row[0]
        for idx in block_indices:
            t_vals = []
            e_vals = []
            for t in timesteps:
                te = [e for e in block_entries if e.block_index == idx and abs(e.timestep - t) < 1e-6]
                if te:
                    t_vals.append(t)
                    e_vals.append(np.mean([e.per_head_entropy.mean().item() for e in te]))
            if t_vals:
                ax.plot(t_vals, e_vals, marker="o", label=f"Block {idx}", markersize=3)
        ax.set_xlabel("Timestep")
        ax.set_ylabel("Mean Entropy")
        ax.set_title(f"{label} — Entropy vs. Timestep")
        ax.legend(fontsize=7)
        
        # I→I quadrant evolution (most relevant for masking)
        ax = ax_row[1]
        for idx in block_indices:
            t_vals = []
            ii_vals = []
            for t in timesteps:
                te = [e for e in block_entries if e.block_index == idx and abs(e.timestep - t) < 1e-6]
                if te:
                    t_vals.append(t)
                    ii_vals.append(np.mean([e.mean_weight_ii for e in te]))
            if t_vals:
                ax.plot(t_vals, ii_vals, marker="o", label=f"Block {idx}", markersize=3)
        ax.set_xlabel("Timestep")
        ax.set_ylabel("Mean I→I Weight")
        ax.set_title(f"{label} — I→I Weight vs. Timestep")
        ax.legend(fontsize=7)
    
    plt.tight_layout()
    plt.show()

plot_temporal_evolution(all_entries)


## 9. Summary Statistics Table

Generate a comprehensive table of profiling metrics for all blocks.

In [ ]:
def print_summary_table(entries):
    """Print a summary table of profiling metrics."""
    print(f"{'Type':<8} {'Block':<6} {'#Entries':<9} {'T→T':>8} {'T→I':>8} {'I→T':>8} {'I→I':>8} "
          f"{'Entropy':>8} {'Sparsity':>9} {'Top-k':>8}")
    print("-" * 95)
    
    for block_type in ["double", "single"]:
        block_entries = [e for e in entries if e.block_type == block_type]
        block_indices = sorted(set(e.block_index for e in block_entries))
        
        for idx in block_indices:
            be = [e for e in block_entries if e.block_index == idx]
            n = len(be)
            tt = np.mean([e.mean_weight_tt for e in be])
            ti = np.mean([e.mean_weight_ti for e in be])
            it = np.mean([e.mean_weight_it for e in be])
            ii = np.mean([e.mean_weight_ii for e in be])
            ent = np.mean([e.per_head_entropy.mean().item() for e in be])
            sp = np.mean([e.per_head_sparsity.mean().item() for e in be])
            tk = np.mean([e.per_head_topk_concentration.mean().item() for e in be])
            print(f"{block_type:<8} {idx:<6} {n:<9} {tt:>8.5f} {ti:>8.5f} {it:>8.5f} {ii:>8.5f} "
                  f"{ent:>8.3f} {sp:>9.4f} {tk:>8.4f}")
        
        if block_type == "double" and block_entries:
            print("-" * 95)

print_summary_table(all_entries)


## 10. Save Profiling Data

Save the profiling data for offline analysis and Phase 2 masking experiments.

In [ ]:
import pickle
import os

# Save profiling entries
output_dir = "2210/profiling_results"
os.makedirs(output_dir, exist_ok=True)

# Save as pickle for full fidelity
save_path = os.path.join(output_dir, "attention_profile_entries.pkl")
with open(save_path, "wb") as f:
    pickle.dump(all_entries, f)
print(f"Saved {len(all_entries)} entries to {save_path}")

# Also save a CSV summary for quick inspection
import csv
csv_path = os.path.join(output_dir, "attention_profile_summary.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "block_type", "block_index", "timestep", "num_text_tokens", "num_image_tokens",
        "mean_weight_tt", "mean_weight_ti", "mean_weight_it", "mean_weight_ii",
        "mean_entropy", "mean_sparsity", "mean_topk_concentration",
    ])
    for e in all_entries:
        writer.writerow([
            e.block_type, e.block_index, e.timestep, e.num_text_tokens, e.num_image_tokens,
            e.mean_weight_tt, e.mean_weight_ti, e.mean_weight_it, e.mean_weight_ii,
            e.per_head_entropy.mean().item(), e.per_head_sparsity.mean().item(),
            e.per_head_topk_concentration.mean().item(),
        ])
print(f"Saved CSV summary to {csv_path}")


## 11. Disable Profiling

Restore standard attention processors for normal inference.

In [ ]:
# Disable profiling and restore standard processors
pipe.transformer.set_profiling_mode(enabled=False)
print("Attention profiling disabled. Standard processors restored.")


## Next Steps (Phase 2)

Based on the profiling results above, identify:
1. Which quadrants show low attention energy → candidates for quadrant masking (Strategy A)
2. Which blocks show high sparsity → candidates for depth-selective masking (Strategy D)
3. Whether I→I attention shows spatial locality → candidate for local window masking (Strategy B)
4. Whether attention patterns vary significantly across timesteps → need for adaptive masking

See `2210/attention_profiling_and_masking_plan.md` for the full experiment plan.